# Run the paper's detector + TTA on songs

Applies this project's published test-time adaptation (TTA) to a folder of songs
(e.g. Indian film / indie / AI-generated tracks) and scores each one for
"AI-generated audio".

**Checkpoint used: `ssl_aasist_wavefake` — [`ash56/ssl-aasist`](https://huggingface.co/ash56/ssl-aasist)**
(XLS-R 300M + AASIST, Garg et al. 2025), loaded through this repo's fairseq-free port
(`vendor_aasist.py` → `aasist_backend.py`). It is the strongest checkpoint this repo
has actually measured, and the only one you can fetch programmatically:

| checkpoint | ITW source EER / AUC | after our TTA |
|---|---|---|
| **`ssl_aasist_wavefake` (ash56)** | **5.04% / 0.986** | **4.06% / 0.990** |
| `deepfense_w2v2_aasist_s240` | 8.41% / 0.970 | 7.88% / 0.973 |
| `deepfense_w2v2_aasist_s2` | 9.59% / 0.966 | 9.06% / 0.970 |
| `deepfense_w2v2_aasist_s42` | 16.41% / 0.918 | 10.10% / 0.961 |

(from `results_public_ckpt.csv`; the paper's *own* source models live in `ckpt_ext/`, which is
not committed to this repo and does not exist on this machine.)

The adaptation config is imported verbatim from `public_ckpt_tta.py` — `Q=0.3`,
`lambda_cons=0.3`, `lr=1e-4`, `E=4`, top-4 transformer blocks' LayerNorms + head,
BatchNorm frozen — so this notebook cannot silently drift from the published method.

---

## Read this before trusting a number

1. **This detector was trained on speech, not music.** Every corpus in the paper
   (ASVspoof, LibriSpeech-TTS, In-the-Wild, ArAD, MLAAD) is a single dry voice.
   Songs add singing, pitch correction, reverb, mastering compression, and a full
   instrumental mix. That is a *larger* domain shift than any gap the paper measures.
   Treat the output as a screening signal, not a verdict.
2. **Ranking transfers, thresholds do not** — the paper's headline finding, and it applies
   here with force. Rank your songs by score and look at the extremes. Do **not** read
   `score > 0.5` as "AI". A usable cut-off needs labelled songs (last section).
3. **TTA needs a mixed pool.** Adaptation pseudo-labels the top 30% of scores as fake and
   the bottom 30% as real. If your folder is all-AI or all-human, that manufactures a
   boundary that does not exist and makes things worse. The TTA cell checks this and
   refuses unless you override.
4. **Strip the instrumental if you can.** Running a vocals-only stem is the single biggest
   quality win, since it puts the input back near the speech domain the model knows:
   `pip install demucs && demucs --two-stems=vocals -o stems songs/*.mp3`, then point
   `SONGS_DIR` at `stems/htdemucs/*/vocals.wav`. Score both ways and compare.

## 1. Config

In [ ]:
from pathlib import Path

REPO       = Path(".").resolve()      # must be the repo root (has public_ckpt_tta.py)
SONGS_DIR  = REPO / "songs"           # <-- put your audio here (mp3/m4a/wav/flac, recursive)
OUT_DIR    = REPO / "songs_out"

CKPT_NAME  = "ssl_aasist_wavefake"    # see table above

# --- chunking: the model eats 64,600 samples (~4.04 s) at 16 kHz -----------------
HOP_FRAC        = 1.0    # 1.0 = non-overlapping windows; 0.5 = 50% overlap (2x cost)
MAX_CHUNKS_SONG = 45     # ~3 min of each song; raise for full-length scoring
SKIP_HEAD_SEC   = 0.0    # skip a fixed intro (e.g. 10.0 to drop label idents)
MIN_RMS         = 1e-3   # drop near-silent windows (they score meaninglessly)

# --- run control -----------------------------------------------------------------
BATCH      = 8           # lower to 4 if you OOM
RUN_TTA    = False       # needs a GPU; ~15-25x the cost of plain scoring
FORCE_TTA  = False       # bypass the mixed-pool gate (read caveat 3 first)
SEED       = 0
INSTALL    = False       # True on a fresh box / Colab

# --- optional labels for calibration ---------------------------------------------
# CSV with columns: file,label   (path or basename; label 1 = AI-generated, 0 = human)
LABELS_CSV = REPO / "songs_labels.csv"

OUT_DIR.mkdir(exist_ok=True)
assert (REPO / "public_ckpt_tta.py").exists(), f"run this from the repo root, not {REPO}"
print("repo:", REPO, "\nsongs:", SONGS_DIR)

## 2. Environment

`ffmpeg` is required for mp3/m4a (`brew install ffmpeg` / `apt-get install -y ffmpeg`).
Without it only wav/flac work.

In [ ]:
if INSTALL:
    %pip install -q torch torchaudio soundfile numpy pandas scikit-learn matplotlib huggingface_hub tqdm

import os, shutil, subprocess, sys, time
import numpy as np, pandas as pd, torch

FFMPEG = shutil.which("ffmpeg")
DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")

print(f"torch {torch.__version__} | device {DEVICE} | ffmpeg {FFMPEG or 'MISSING (wav/flac only)'}")
if DEVICE == "cpu":
    print("WARNING: XLS-R 300M on CPU is ~1-2 s per 4 s window. Scoring is slow; TTA is not viable.")
if RUN_TTA and DEVICE != "cuda":
    print(f"WARNING: RUN_TTA on {DEVICE}. Expect hours. A Colab T4 does this in minutes.")

torch.manual_seed(SEED); np.random.seed(SEED)

## 3. Fetch the checkpoint and build the fairseq-free backend

Idempotent — skips anything already on disk. ~1.2 GB download on first run.

In [ ]:
from huggingface_hub import hf_hub_download

CKPT_DIR = REPO / "public_ckpt" / "ssl_aasist"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

for fn in ("pytorch_model.bin", "model_hf.py", "config.json", "config_ssl.py"):
    dst = CKPT_DIR / fn
    if dst.exists():
        print(f"have {fn}"); continue
    src = hf_hub_download("ash56/ssl-aasist", fn)
    shutil.copy(src, dst)
    print(f"fetched {fn} ({dst.stat().st_size/1e6:.0f} MB)")

# vendor_aasist.py rewrites their model_hf.py into aasist_backend.py, swapping the
# fairseq XLS-R front-end for torchaudio's. Every substitution is asserted, so an
# upstream edit fails loudly instead of vendoring something different.
if not (REPO / "aasist_backend.py").exists():
    print(subprocess.run([sys.executable, "vendor_aasist.py"], cwd=REPO,
                         capture_output=True, text=True, check=True).stdout)
else:
    print("have aasist_backend.py")

In [ ]:
sys.path.insert(0, str(REPO))
import public_ckpt_tta as pct   # published TTA config + scoring/adaptation, imported not copied

CFG  = pct.CHECKPOINTS[CKPT_NAME]
CROP = CFG["crop"]
SR   = pct.SR
HOP  = max(1, int(CROP * HOP_FRAC))

print(f"{CKPT_NAME}: {CFG['arch']}, trained on {CFG['train_data']}")
print(f"crop {CROP} samples ({CROP/SR:.2f} s @ {SR} Hz), hop {HOP}, P(fake) = softmax col {CFG['fake_col']}")
print(f"TTA config: Q={pct.Q} lambda_cons={pct.LAMBDA_CONS} lr={pct.TTA_LR} "
      f"E={pct.TTA_EPOCHS} n_finetune={pct.N_FINETUNE} anchor={pct.USE_ANCHOR}")

t0 = time.time()
model, n_front, n_back = pct.build_model(CKPT_NAME, DEVICE)
print(f"loaded {n_front} frontend + {n_back} backend tensors in {time.time()-t0:.0f}s")

## 4. Decode songs → 4 s windows

Each song is decoded once to 16 kHz mono, cut into windows, and cached to
`songs_out/chunks.npy` so re-running the notebook is instant.

In [ ]:
import soundfile as sf

AUDIO_EXT = {".mp3", ".m4a", ".wav", ".flac", ".ogg", ".opus", ".aac", ".wma", ".webm"}


def decode_16k_mono(path):
    """Any container -> float32 mono at 16 kHz. ffmpeg when available (songs are
    mp3/m4a at 44.1 kHz stereo, which libsndfile will not touch), else soundfile."""
    if FFMPEG:
        p = subprocess.run([FFMPEG, "-v", "error", "-i", str(path),
                            "-f", "f32le", "-ac", "1", "-ar", str(SR), "-"],
                           capture_output=True)
        if p.returncode != 0:
            raise RuntimeError(f"ffmpeg failed on {path}: {p.stderr.decode()[:200]}")
        return np.frombuffer(p.stdout, dtype=np.float32).copy()
    x, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if x.ndim > 1:
        x = x.mean(axis=1)
    if sr != SR:
        import torchaudio
        x = torchaudio.functional.resample(torch.from_numpy(x), sr, SR).numpy()
    return x


def window(x):
    """Windows of CROP samples. Short clips are tile-repeated, matching upstream's pad()."""
    x = x[int(SKIP_HEAD_SEC * SR):]
    if len(x) == 0:
        return []
    if len(x) < CROP:
        return [np.tile(x, int(CROP / len(x)) + 1)[:CROP].astype(np.float32)]
    out = []
    for s in range(0, len(x) - CROP + 1, HOP):
        w = x[s:s + CROP]
        if float(np.sqrt(np.mean(w ** 2))) >= MIN_RMS:   # skip silence/fade
            out.append(w.astype(np.float32))
        if len(out) >= MAX_CHUNKS_SONG:
            break
    return out


files = sorted(p for p in SONGS_DIR.rglob("*") if p.suffix.lower() in AUDIO_EXT)
assert files, f"no audio under {SONGS_DIR} — create it and drop songs in"
print(f"{len(files)} audio files under {SONGS_DIR}")

In [ ]:
CHUNK_NPY, CHUNK_IDX = OUT_DIR / "chunks.npy", OUT_DIR / "chunks_index.csv"

if CHUNK_NPY.exists() and CHUNK_IDX.exists():
    chunks = np.load(CHUNK_NPY, mmap_mode="r")
    index = pd.read_csv(CHUNK_IDX)
    print(f"reusing cache: {len(index)} windows from {index.file.nunique()} songs "
          f"(delete {CHUNK_NPY.name} to rebuild)")
else:
    all_w, rows, t0 = [], [], time.time()
    for i, p in enumerate(files):
        try:
            ws = window(decode_16k_mono(p))
        except Exception as e:
            print(f"  SKIP {p.name}: {e}"); continue
        if not ws:
            print(f"  SKIP {p.name}: no non-silent window"); continue
        for j, w in enumerate(ws):
            rows.append(dict(file=str(p.relative_to(SONGS_DIR)), chunk=j,
                             t_start=round(SKIP_HEAD_SEC + j * HOP / SR, 2)))
        all_w.extend(ws)
        if (i + 1) % 10 == 0 or i + 1 == len(files):
            print(f"  decoded {i+1}/{len(files)} files, {len(all_w)} windows ({time.time()-t0:.0f}s)")
    chunks = np.stack(all_w)
    index = pd.DataFrame(rows)
    np.save(CHUNK_NPY, chunks); index.to_csv(CHUNK_IDX, index=False)
    del all_w

buf = torch.from_numpy(np.asarray(chunks)).half()      # fp16 host cache, batches move per step
all_idx = torch.arange(len(index))
print(f"{len(index)} windows / {index.file.nunique()} songs, buffer {buf.numel()*2/1e9:.2f} GB")
print(f"median windows per song: {index.groupby('file').size().median():.0f}")

## 5. Score with the unadapted (source) model

In [ ]:
t0 = time.time()
s_src = pct.score(model, buf, all_idx, CFG["fake_col"], BATCH, DEVICE)
print(f"scored {len(s_src)} windows in {(time.time()-t0)/60:.1f} min")
index["score_src"] = s_src


def per_song(index, col):
    """Window scores -> one row per song.

    `mean` is the headline; `p90`/`frac_hi` catch a song where only the vocal
    sections look synthetic, which a mean over instrumental-heavy windows buries.
    """
    g = index.groupby("file")[col]
    out = pd.DataFrame({
        "n_win":   g.size(),
        "mean":    g.mean(),
        "median":  g.median(),
        "p90":     g.quantile(0.9),
        "max":     g.max(),
        "frac_hi": g.apply(lambda s: float((s >= 0.5).mean())),
    }).sort_values("mean", ascending=False)
    out.insert(0, "rank", range(1, len(out) + 1))
    return out


songs = per_song(index, "score_src")
songs.to_csv(OUT_DIR / "songs_scores.csv")
index.to_csv(OUT_DIR / "window_scores.csv", index=False)
print(f"\nwrote {OUT_DIR/'songs_scores.csv'} — ranked most→least AI-looking\n")
songs.head(20).round(3)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(songs["mean"], bins=min(30, max(5, len(songs) // 2)), color="#4472a8", edgecolor="w")
ax[0].set(xlabel="song mean P(fake)", ylabel="songs", title="Song-level score distribution")
ax[0].axvline(0.5, ls="--", c="crimson", label="0.5 (NOT a valid threshold)")
ax[0].legend(fontsize=8)
ax[1].hist(index["score_src"], bins=50, color="#7a9cc6", edgecolor="w")
ax[1].set(xlabel="window P(fake)", ylabel="windows", title="Window-level score distribution")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_song_scores.png", dpi=140); plt.show()

# Two clean modes = the detector is separating something. One blob = it is not
# discriminating on this material, and no threshold will rescue that.
print(f"song mean P(fake): min {songs['mean'].min():.3f}  median {songs['mean'].median():.3f}  "
      f"max {songs['mean'].max():.3f}  spread {songs['mean'].max()-songs['mean'].min():.3f}")

### Where in a song does it fire?

Per-window trace for the top and bottom song. A trace that tracks the vocal sections
is evidence the detector is looking at the voice; one that tracks loudness or the drop
is evidence it is reacting to the mix.

In [ ]:
picks = list(songs.index[:1]) + list(songs.index[-1:])
fig, ax = plt.subplots(len(picks), 1, figsize=(11, 2.4 * len(picks)), squeeze=False)
for a, f in zip(ax[:, 0], picks):
    d = index[index.file == f]
    a.plot(d.t_start, d.score_src, marker="o", ms=3, lw=1)
    a.axhline(0.5, ls="--", c="crimson", lw=0.8)
    a.set(ylim=(-0.02, 1.02), ylabel="P(fake)", xlabel="time (s)",
          title=f"{f}  (mean {songs.loc[f,'mean']:.3f})")
plt.tight_layout(); plt.savefig(OUT_DIR / "fig_song_traces.png", dpi=140); plt.show()

## 6. Test-time adaptation (the paper's method)

`pct.adapt` is the published loop, unmodified: score the pool → pseudo-label the
confident top/bottom 30% → self-train the top-4 blocks' LayerNorms + head → enforce
consistency under a channel perturbation. **No labels are used.**

The gate below refuses to run on a one-sided pool, where the quantile split would invent
a fake/real boundary inside a single class.

In [ ]:
def pool_is_mixed(s, edge=0.85):
    """Cheap sanity check that the pool plausibly spans both classes."""
    hi, lo = float((s >= 0.5).mean()), float((s < 0.5).mean())
    q30, q70 = np.quantile(s, 0.3), np.quantile(s, 0.7)
    return (max(hi, lo) < edge) and (q70 - q30 > 0.05), dict(
        frac_hi=round(hi, 3), q30=round(float(q30), 3), q70=round(float(q70), 3))


ok, diag = pool_is_mixed(s_src)
print(f"mixed-pool check: {'PASS' if ok else 'FAIL'}  {diag}")

if not RUN_TTA:
    print("RUN_TTA is False — skipping adaptation (source scores above are your result).")
elif not (ok or FORCE_TTA):
    print("Refusing to adapt: the pool looks one-sided, so pseudo-labels would split a\n"
          "single class in half. Add songs of the other kind, or set FORCE_TTA=True to override.")
else:
    t0 = time.time()
    model, info = pct.adapt(model, buf, all_idx, CFG["fake_col"], BATCH, DEVICE,
                            use_st=True, use_cons=True)
    s_tta = pct.score(model, buf, all_idx, CFG["fake_col"], BATCH, DEVICE)
    index["score_tta"] = s_tta
    songs_tta = per_song(index, "score_tta")
    songs_tta.to_csv(OUT_DIR / "songs_scores_tta.csv")
    index.to_csv(OUT_DIR / "window_scores.csv", index=False)

    rho = index[["score_src", "score_tta"]].corr(method="spearman").iloc[0, 1]
    print(f"\nadapted in {(time.time()-t0)/60:.1f} min | {info['n_trainable_params']:,} trainable params")
    print(f"window-score Spearman rho (source vs adapted): {rho:.3f}")
    print("  rho near 1.0 = adaptation only rescaled scores, the ranking is unchanged.")
    print("  rho well below 1.0 = it genuinely reordered clips; without labels you cannot")
    print("  tell whether that reordering is an improvement. Use the labelled section below.")
    display(songs_tta.head(20).round(3))

## 7. Calibration and measurement (needs labels)

Everything above is a *ranking*. To get a threshold, or to know whether the detector
works at all on your material, label a subset: a CSV `songs_labels.csv` with columns
`file,label` (`file` = path relative to `SONGS_DIR`, or just the basename; `label` = 1 for
AI-generated, 0 for human). Twenty of each is enough for a first read.

This cell reports EER/AUC and the EER threshold — the same metrics the paper is judged on.

In [ ]:
from sklearn.metrics import roc_auc_score
from metrics import compute_eer

if not LABELS_CSV.exists():
    print(f"no {LABELS_CSV.name} — skipping. Without it, use ranks only, never a threshold.")
else:
    lab = pd.read_csv(LABELS_CSV)
    by_base = {Path(f).name: int(l) for f, l in zip(lab.file, lab.label)}
    scored = songs.copy()
    scored["label"] = [by_base.get(Path(f).name, by_base.get(f, -1)) for f in scored.index]
    scored = scored[scored.label >= 0]
    assert len(scored) and scored.label.nunique() == 2, \
        f"need both classes among matched songs, got {len(scored)} rows"
    print(f"matched {len(scored)} labelled songs "
          f"({int((scored.label==0).sum())} human, {int((scored.label==1).sum())} AI)\n")

    for col in ("score_src", "score_tta"):
        if col == "score_tta" and "score_tta" not in index.columns:
            continue
        agg = per_song(index, col).loc[scored.index]
        for stat in ("mean", "p90", "max"):
            y, s = scored.label.values, agg[stat].values
            eer, thr = compute_eer(y, s)
            print(f"{col:10s} {stat:6s}  EER {eer*100:5.2f}%  AUC {roc_auc_score(y, s):.4f}  "
                  f"EER threshold {thr:.3f}")
        print()

    print("Polarity check: AI songs must score HIGHER than human ones.")
    m0, m1 = scored[scored.label == 0]["mean"].mean(), scored[scored.label == 1]["mean"].mean()
    print(f"  mean(human) {m0:.3f}  vs  mean(AI) {m1:.3f}  ->  "
          f"{'OK' if m1 > m0 else 'INVERTED — flip CFG[fake_col] and re-score'}")
    print("\nAUC ~0.5 means the detector does not transfer to this material; a threshold")
    print("cannot fix that. AUC high but EER threshold far from 0.5 is the paper's")
    print("expected case: ranking transfers, the decision boundary does not — recalibrate")
    print("on this threshold and re-check it whenever the music domain changes.")

## Outputs

| file | what |
|---|---|
| `songs_out/songs_scores.csv` | per-song source scores, ranked most→least AI-looking |
| `songs_out/songs_scores_tta.csv` | same after test-time adaptation (if run) |
| `songs_out/window_scores.csv` | per-4 s-window scores with timestamps |
| `songs_out/fig_song_scores.png` | score distributions |
| `songs_out/fig_song_traces.png` | per-window trace, top and bottom song |
| `songs_out/chunks.npy` | decoded window cache — delete to rebuild |

**How to use it honestly.** Rank, inspect the extremes, and label those by ear. Once you
have ~40 labelled songs, section 7 tells you whether the detector separates your material
at all (AUC) and where the cut-off sits (EER threshold). If AUC ≈ 0.5, the answer is that
a speech-trained detector does not transfer to full-mix music — a real, reportable result,
and the point at which vocal separation or fine-tuning on song data becomes the next step.